In [2]:
import os
import sys
sys.path.append('/host/d/Github')  ### remove this if not needed!
import numpy as np
import pandas as pd
import random
import torch
from tqdm import tqdm
import torch.backends.cudnn as cudnn
import argparse
 
from SAM_echo_xjtlu.utils.model_util import *
from SAM_echo_xjtlu.segment_anything.model import build_model 
from SAM_echo_xjtlu.utils.save_utils import *
from SAM_echo_xjtlu.utils.config_util import Config

import SAM_echo_xjtlu.inference_engine as inference_engine

import SAM_echo_xjtlu.functions_collection as ff
import SAM_echo_xjtlu.Data_processing as Data_processing
import SAM_echo_xjtlu.Build_lists.Build_list as Build_list
import SAM_echo_xjtlu.data_loader.generator as generator

main_path = '/host/d/projects/FETUS_competition/' ### change to your main path

### step 1: define parameters for this experiment

In [3]:
trial_name = 'trial_sam-15'
output_dir = os.path.join(main_path, 'models', trial_name) # change to your output dir
ff.make_folder([output_dir])


# define the pretrained model
pretrained_model = os.path.join(output_dir, 'models/model-150.pth')


In [4]:
# many important parameters, focus on ones that I comment with ###!! 
# most of them should be set the same as in training

def get_args_parser(img_size = 128, num_classes = 3, pretrained_model = None, original_sam = None, start_epoch = None, total_training_epochs = 1000, save_model_every = 1,  vit_type = "vit_h"):
    parser = argparse.ArgumentParser('SAM fine-tuning', add_help=True)

    # img size
    parser.add_argument('--img_size', default=img_size, type=int)  ## !!

    ## augmentation
    parser.add_argument('--augment_frequency', default= 0.8, type=float) ## !!

    ## segmentation classes
    parser.add_argument('--num_classes', type=int, default=num_classes) ## !!

    ## pretrained sam
    parser.add_argument('--resume', default = original_sam) ##!!

    # for training
    parser.add_argument('--total_training_epochs', default = total_training_epochs, type=int)
    parser.add_argument('--accum_iter', default=20, type=int, help='Accumulate gradient iterations (for increasing the effective batch size under memory constraints)') ##!!
    parser.add_argument('--print_freq', default=10, type = int) 
    parser.add_argument('--save_model_file_every_N_epoch', default=save_model_every, type = int)  ## !!
    parser.add_argument('--batch_size', default=1, type=int, help='Batch size per GPU (effective batch size is batch_size * accum_iter * # gpus')  ## !!
    parser.add_argument('--weight_decay', type=float, default=0.05, help='weight decay (default: 0.05)')
    parser.add_argument('--lr', type=float, default=1e-4, metavar='LR', help='base learning rate: absolute_lr = base_lr * total_batch_size / 256') ## !!
    parser.add_argument('--lr_update_every_N_epoch', default=100, type = int) ## !!
    parser.add_argument('--lr_decay_gamma', default=0.95)
    parser.add_argument('--warmup_epochs', type=int, default=10, metavar='N', help='epochs to warmup LR')
    parser.add_argument('--loss_weights', default = [0,1] )  #### !! weighting for loss function [BCE, Dice]

    if start_epoch == None:
        parser.add_argument('--start_epoch', default=1, type=int, metavar='N', help='start epoch')
    else:
        parser.add_argument('--start_epoch', default= start_epoch, type=int, metavar='N', help='start epoch')

    # standard
    parser.add_argument('--text_prompt', default = False)
    parser.add_argument('--box_prompt', default= False) 
    parser.add_argument('--pretrained_model', default = pretrained_model)
    
    parser.add_argument('--validation', default=False) ## !!
    parser.add_argument('--cross_frame_attention', default=False) # False

    parser.add_argument('--model_type', type=str, default='sam')
    parser.add_argument('--n_gpu', type=int, default=1, help='total gpu') 
    parser.add_argument('--use_amp', action='store_true', help='If activated, adopt mixed precision for acceleration')
    parser.add_argument("--config", help="Path to the training config file.", default="configs/config.yaml",)

    parser.add_argument('--seed', default=1234, type=int)   
    parser.add_argument('--input_type', type=str, default='2DT') #has to be 2DT
    parser.add_argument('--vit_type', type=str, default=vit_type)
    parser.add_argument('--max_timeframe', default=1, type=int) 
                        

    parser.add_argument('--turn_zero_seg_slice_into', default=10, type=int)
 
    return parser

# define the original sam model weights (you should download it from online to your local path)
original_sam = '/host/d/Data/pretrained_SAM_weights/sam_vit_h.pth'  # change to your original sam model path

args = get_args_parser(img_size = 128, ## important !! need to change based on your dataset
        num_classes = 3, ## important !! need to change based on your dataset
        pretrained_model = pretrained_model, 
        original_sam = original_sam,  
        vit_type = "vit_h")

args = args.parse_args([])

# some other settings
cfg = Config(args.config)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
cudnn.benchmark = True


### step 2: define data you want to predict on and corresponding data generator


In [5]:
# change the excel path to your own path
patient_list_spreadsheet = os.path.join('/host/d/Github/SAM_echo_xjtlu/example_data/data/Patient_list','patient_list.xlsx')
build_sheet =  Build_list.Build(patient_list_spreadsheet)

_, patient_id_list_pred, img_file_list_pred, seg_file_list_pred = build_sheet.__build__(index_list=[0])  
print('all pred img files:', img_file_list_pred)

all pred img files: ['/host/d/Github/SAM_echo_xjtlu/example_data/data/example_0001/img.nii.gz']


In [6]:
### get the data generator for prediction
# define this generator
generator_pred = generator.Dataset_CMR( 
    image_file_list = img_file_list_pred,
    seg_file_list = seg_file_list_pred,

    image_shape = [args.img_size, args.img_size],
    center_crop_according_to_which_class  = [1], #default: crop according to class 1 (LV)

    shuffle = False,
    image_normalization = True,
    augment = False, # no augmentation for prediction
    augment_frequency = -1, ### not used
    )

### predict

In [7]:
data_loader_pred = torch.utils.data.DataLoader(generator_pred, batch_size = 1, shuffle = False, pin_memory = True, num_workers = 0)

model = build_model(args, device)#skip_nameing = True, chunk = np.shape(np.zeros(0)))

# load the pretrained model
if args.pretrained_model is not None:
    print('loading pretrained model : ', args.pretrained_model)
    finetune_checkpoint = torch.load(args.pretrained_model)
    model.load_state_dict(finetune_checkpoint["model"])
else:
    ValueError('no pretrained model specified for prediction!')

Important! text prompt: False
Important! box prompt: False
loading pretrained model :  /host/d/projects/FETUS_competition/models/trial_sam-15/models/model-150.pth


In [ ]:
data_loader_pred = torch.utils.data.DataLoader(generator_pred, batch_size = 1, shuffle = False, pin_memory = True, num_workers = 0)
with torch.no_grad():
    with torch.cuda.amp.autocast():
                            
        # do the prediction for each slice (2D+T) one by one
        for data_iter_step, batch in tqdm(enumerate(data_loader_pred)):

            image_filename = batch["image_filename"][0]
            patient_id = os.path.basename(os.path.dirname(image_filename))
            
            print('patient_id: ', patient_id)
                
            save_folder_patient = os.path.join(output_dir, 'predictions', patient_id)
            ff.make_folder([os.path.dirname(save_folder_patient), save_folder_patient])

            batch["image"]= batch["image"].cuda()
                    
            output = model(batch, args.img_size)


            torch.cuda.synchronize()
            
            inference_engine.save_predictions1( batch = batch, output = output, args = args, save_folder_patient = save_folder_patient)

1it [00:00,  5.24it/s]

processsed_image shape: (128, 128, 15)
patient_id:  example_0001
original shape:  (216, 256)
